Hier wird analysiert und dargestellt das Einbeziehen welcher features denn wie viele samples liefert.

In [13]:
import pandas as pd
import numpy as np

# =========================================================
# 1. Daten laden & bereinigen (wie im Haupt-Skript)
# =========================================================
file_path = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\allValues_Features_PM25_NDVI_TPI_S5P_ERA5_MAIAC_Verkehr_Wetter.csv"
df = pd.read_csv(file_path, sep=";", decimal=",")

df['station_id'] = df['station_id'].astype(str).str.strip()

# Kaputte / verzerrende Sensoren vorab entfernen
bad_sensors = ['5456', '15449', '46505', '9094']
df = df[~df['station_id'].isin(bad_sensors)].copy()

# =========================================================
# 2. Szenarien (Feature-Sets) definieren
# =========================================================
scenarios = [
    {
        "Name": "Scenario 1 (Only ICON-D2 & ERA5 data)",
        "Features": ['icon_temp_2m', 'icon_wind_speed', 'icon_wind_direction', 'ERA5_BLH']
    },
    {
        "Name": "Scenario 2 (only traffic)",
        "Features": [
                     'IDW_car', 'IDW_truck', 'prox_kumulativ_car', 'prox_kumulativ_truck']
    },
    {
        "Name": "Scenario 3 (only Sentinel-5P data)",
        "Features": ['S5P_NO2', 'S5P_UVAI']
    },
    {
        "Name": "Szenario 4 (only MODIS data)",
        "Features": [ 'MAIAC_AOD']
    },
    {
        "Name": "Szenario 5 (all Features)",
        "Features": ['icon_temp_2m', 'S5P_NO2', 'S5P_UVAI', 'ERA5_BLH',
                     'IDW_car', 'IDW_truck', 'prox_kumulativ_car', 'prox_kumulativ_truck',
                     'icon_wind_speed', 'icon_wind_direction',
                     'ndvi', 'tpi_300m', 'tpi_1500m', 'icon_precip_12h', "MAIAC_AOD"]
    }
]

# =========================================================
# 3. Schleife: Sample Size berechnen
# =========================================================
print("Berechne verfügbare Samples pro Parameter-Szenario...\n")

results = []
target = 'PM25_Target'

for sc in scenarios:
    # Wir droppen alle Zeilen, die im Target ODER in den aktuellen Features ein NaN haben
    cols_to_check = [target] + sc["Features"]
    df_filtered = df.dropna(subset=cols_to_check)

    # Speichern der Ergebnisse
    results.append({
        "Szenario": sc["Name"],
        "Verwendete Parameter": ", ".join(sc["Features"]),
        "Verfügbare Samples": len(df_filtered)
    })

# =========================================================
# 4. Als saubere Tabelle ausgeben
# =========================================================
results_df = pd.DataFrame(results)

# Optional: Speichere die Tabelle für deine Thesis ab
output_csv = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Diagrams\Sample_Size_Scenarios.csv"
results_df.to_csv(output_csv, sep=";", index=False, encoding='utf-8-sig')

# Tabelle in der Konsole anzeigen
pd.set_option('display.max_colwidth', None) # Damit die Parameter-Spalte nicht abgeschnitten wird
print(results_df.to_string(index=False))
print(f"\n-> Die Übersicht wurde auch hier gespeichert: {output_csv}")

Berechne verfügbare Samples pro Parameter-Szenario...

                             Szenario                                                                                                                                                                                 Verwendete Parameter  Verfügbare Samples
Scenario 1 (Only ICON-D2 & ERA5 data)                                                                                                                                         icon_temp_2m, icon_wind_speed, icon_wind_direction, ERA5_BLH                 572
            Scenario 2 (only traffic)                                                                                                                                         IDW_car, IDW_truck, prox_kumulativ_car, prox_kumulativ_truck                 572
   Scenario 3 (only Sentinel-5P data)                                                                                                                                               